# 08 Synthesis

Assembles one scenario table following the intended chain (terrain/lifts to capacity, snow reliability adjusting capacity, capacity to workforce, workforce to housing demand, land capacity for supply) from steps 03-07's own data/processed outputs. Two links in that chain have no evidence-backed number yet (snow-adjusted capacity, and land capacity in units), so this notebook carries them as explicit not-available markers rather than omitting them silently or filling them with a guess.

In [ ]:
processed_dir = "data/processed"

## Capacity and workforce, by phase (steps 03 and 05)

Reads step 03's CCC-reproduction summary (the appendix's stated totals, confirmed reproducible from per-lift data) and step 05's employee/housing-demand estimates, joined on phase.

In [ ]:
import pandas as pd

ccc = pd.read_csv(f"{processed_dir}/03_ccc_phase_reproduction_summary.csv")
employees = pd.read_csv(f"{processed_dir}/05_employee_estimates.csv")
scenario = employees.merge(
    ccc[["phase", "ccc_skiers"]], on="phase", suffixes=("", "_reproduced_check")
)
scenario

## Where the chain is incomplete (steps 04 and 07)

Snow reliability (step 04) built elevation bands and a historical baseline but did not evaluate a reliability indicator against any climate projection, so there is no snow-adjusted capacity number to apply here. Land capacity (step 07) found real zoning and parcel data but not the bylaw density figures needed to state a unit-capacity number, so there is no supply-side figure to compare housing demand against yet. Both are recorded as explicit gaps, at the scenario level, not silently dropped.

In [ ]:
scenario["snow_adjusted_ccc"] = pd.NA  # step 04: no projection evaluated yet
scenario["land_capacity_units"] = pd.NA  # step 07: no bylaw density data yet
scenario

## Write outputs

One scenario table, phase by low/mid/high employee and housing-demand bands, with the two open links marked explicitly.

In [ ]:
import os

os.makedirs(processed_dir, exist_ok=True)
scenario.to_csv(f"{processed_dir}/08_scenario_table.csv", index=False, encoding="utf-8")
print("wrote 08_scenario_table.csv")

## Checks

The CCC figures joined from step 03 must match step 05's own CCC-by-phase column exactly (both ultimately come from claim C025, so a mismatch would mean the two notebooks drifted apart), and the two open-gap columns must still be null, not accidentally filled with a placeholder number.

In [ ]:
assert (scenario["ccc_skiers"] == scenario["ccc_skiers_reproduced_check"]).all()
assert scenario["snow_adjusted_ccc"].isna().all()
assert scenario["land_capacity_units"].isna().all()
print("checks passed")

## Versions

In [ ]:
import importlib.metadata
import sys

print("python", sys.version)
for pkg in ["pandas"]:
    print(pkg, importlib.metadata.version(pkg))